# Práctica guiada — Inferencia y contratos

Este notebook es para el alumnado. Completa las celdas marcadas con `TODO` y responde las preguntas antes de mirar la solución docente.

**Objetivo:** pasar de una fila CSV a una especificación clara para el módulo de inferencia de la clase 2.

## Cómo trabajar

1. Copia esta carpeta como tu espacio de trabajo.
2. Ejecuta `uv sync` desde `problem/starter/` cuando empiece la clase 2.
3. En esta práctica no implementes todavía el CLI completo.
4. Trabaja en parejas y cambia quién escribe cada diez minutos.
5. No abras `solutions/` hasta el debrief.

In [ ]:
from dataclasses import dataclass
from typing import Literal
import csv
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field, ValidationError

DATA_PATH = Path("../../../../../../assets/03-wine-quality/inference_samples.csv")
print(DATA_PATH)
print(DATA_PATH.exists())

## 1. Leer los datos nuevos

### Práctica A — Inspección del CSV

Completa la celda siguiente para:

- Leer el CSV con `csv.DictReader`.
- Imprimir el número de filas.
- Imprimir las columnas.
- Guardar la primera fila en una variable llamada `row`.

In [ ]:
# Lectura del CSV de datos nuevos (sin etiqueta).
with DATA_PATH.open(newline="", encoding="utf-8") as source:
    reader = csv.DictReader(source)
    columns = list(reader.fieldnames or [])
    rows = list(reader)

row = rows[0]

print("Filas:", len(rows))
print("Columnas:", columns)
print("Primera fila:", row)

assert len(rows) == 5
assert len(columns) == 12 and columns[0] == "sample_id"
assert all(isinstance(value, str) for value in row.values())  # DictReader sólo da texto

### Preguntas

1. ¿Qué campo identifica la muestra pero no es una característica del modelo?
2. ¿Cuántas características numéricas hay?
3. ¿Qué columna extra añadirías para probar el contrato?
4. ¿Qué valor de `ph` sería claramente inválido?

**Respuestas:**

1. `sample_id` (`red-001` … `red-005`). Sirve para trazar qué muestra se ha predicho y devolverla en la salida, pero no entra en el vector del modelo.
2. 11 características numéricas: las 12 columnas del CSV menos `sample_id`.
3. `unexpected_field`, o mejor aún `quality`: la etiqueta no debe llegar en inferencia y, si llega, el contrato tiene que rechazarla en vez de ignorarla.
4. `ph = 99` (o cualquier valor negativo). El contrato admite 2,5–4,5 y en vino tinto lo normal está entre 3 y 3,6.

## 2. Construir el contrato con Pydantic

El contrato del modelo contiene estas características:

| Campo | Rango orientativo |
|---|---|
| `fixed_acidity` | 0–20 |
| `volatile_acidity` | 0–2 |
| `citric_acid` | 0–2 |
| `residual_sugar` | 0–20 |
| `chlorides` | 0–1 |
| `free_sulfur_dioxide` | 0–100 |
| `total_sulfur_dioxide` | 0–300 |
| `density` | 0.98–1.01 |
| `ph` | 2.5–4.5 |
| `sulphates` | 0–3 |
| `alcohol` | 5–20 |

In [ ]:
class WineQualityRequestPydantic(BaseModel):
    # extra="forbid": una columna que no está en el contrato es un error, no se ignora.
    model_config = ConfigDict(extra="forbid")

    fixed_acidity: float = Field(ge=0, le=20)
    volatile_acidity: float = Field(ge=0, le=2)
    citric_acid: float = Field(ge=0, le=2)
    residual_sugar: float = Field(ge=0, le=20)
    chlorides: float = Field(ge=0, le=1)
    free_sulfur_dioxide: float = Field(ge=0, le=100)
    total_sulfur_dioxide: float = Field(ge=0, le=300)
    density: float = Field(ge=0.98, le=1.01)
    ph: float = Field(ge=2.5, le=4.5)
    sulphates: float = Field(ge=0, le=3)
    alcohol: float = Field(ge=5, le=20)


# sample_id identifica la muestra, pero no es una característica del modelo.
sample_id = row["sample_id"]
model_row = {name: value for name, value in row.items() if name != "sample_id"}
request = WineQualityRequestPydantic.model_validate(model_row)

print(sample_id, request)
assert isinstance(request.alcohol, float)  # Pydantic convierte "9.4" en 9.4

### Práctica B — Validación

Completa una prueba para cada caso:

- Una fila válida.
- Una columna `unexpected_field`.
- Un `ph` fuera de rango.

Antes de ejecutar, escribe qué error esperas.

In [ ]:
# Errores esperados ANTES de ejecutar:
# - fila válida: ningún error, todos los campos convertidos a float.
# - unexpected_field: 1 error de tipo extra_forbidden en "unexpected_field".
# - ph = 99: 1 error de tipo less_than_equal en "ph" (debe ser <= 4.5).

def validate_case(case_name, candidate):
    print(f"--- {case_name}")
    try:
        validated = WineQualityRequestPydantic.model_validate(candidate)
    except ValidationError as error:
        print(error)
        return [item["type"] for item in error.errors()]
    print("Válida:", validated.model_dump())
    return []


assert validate_case("fila válida", model_row) == []
assert validate_case(
    "columna extra", {**model_row, "unexpected_field": "not allowed"}
) == ["extra_forbidden"]
assert validate_case("ph fuera de rango", {**model_row, "ph": "99"}) == ["less_than_equal"]

## 3. Comparar con `dataclass`

Construye la misma estructura con `dataclass`. No añadas todavía validación manual: observa qué garantiza Python y qué no garantiza.

In [ ]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
    fixed_acidity: float
    volatile_acidity: float
    citric_acid: float
    residual_sugar: float
    chlorides: float
    free_sulfur_dioxide: float
    total_sulfur_dioxide: float
    density: float
    ph: float
    sulphates: float
    alcohol: float


numeric_row = {name: float(value) for name, value in model_row.items()}
valid_dc = WineQualityRequestDataclass(**numeric_row)
print("Válida:", valid_dc)

# ph = 99 se acepta sin ningún error: dataclass no valida rangos.
ph_99 = WineQualityRequestDataclass(**{**numeric_row, "ph": 99.0})
print("ph=99 aceptado:", ph_99.ph)

# Tampoco convierte tipos: con la fila del CSV los valores siguen siendo texto.
raw_dc = WineQualityRequestDataclass(**model_row)
print("Sin conversión, alcohol es:", type(raw_dc.alcohol).__name__)

# Lo único que sí garantiza es la lista de campos: un argumento desconocido da TypeError.
try:
    WineQualityRequestDataclass(**numeric_row, unexpected_field="not allowed")
except TypeError as error:
    print("Campo extra:", error)

assert ph_99.ph == 99.0 and isinstance(raw_dc.alcohol, str)

### Decisión de diseño

Completa la frase:

> Usaría Pydantic en **la frontera del sistema (lectura del CSV y, más adelante, la API)** porque **convierte los tipos, valida rangos y campos extra y devuelve errores legibles que indican qué campo falla**. Usaría `dataclass` en **estructuras internas que ya han pasado el contrato, como el vector de características entre `preprocess` e `inference`,** porque **es ligera, con `frozen=True` es inmutable y no tiene sentido volver a validar datos que ya están validados**.

Como muestra la celda anterior, `dataclass` acepta `ph = 99` y deja los valores como texto: sólo comprueba que los nombres de los campos existen.

## 4. Definir el orden del vector

El modelo no recibe un diccionario. Recibe una lista numérica con un orden estable.

In [ ]:
# Orden exacto del vector: el del CSV y el del contrato, sin sample_id.
FEATURE_NAMES = (
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
)

vector = [getattr(request, name) for name in FEATURE_NAMES]
print(vector)

assert len(FEATURE_NAMES) == 11
assert list(FEATURE_NAMES) == [name for name in columns if name != "sample_id"]
assert list(FEATURE_NAMES) == list(WineQualityRequestPydantic.model_fields)
assert all(isinstance(value, float) for value in vector)

### Práctica C — Error de orden

Intercambia mentalmente `density` y `alcohol`. Responde:

- ¿El vector sigue teniendo 11 valores?
- ¿El clasificador detectaría necesariamente el error?
- ¿Cómo evitarías que cada script definiera un orden distinto?

**Respuestas:**

- Sí. Siguen siendo 11 floats, así que cualquier comprobación de longitud o de tipo pasa.
- No. El modelo recibe una lista de números sin nombres: tomaría `0.9978` como alcohol y `9.4` como densidad y devolvería una predicción sin ningún error, probablemente equivocada. La validación de rangos tampoco lo detecta, porque se hace antes, sobre el diccionario con nombres.
- Definiendo `FEATURE_NAMES` una sola vez (en `preprocess.py`) e importándolo desde el resto de módulos, guardando la misma lista en el artefacto (`feature_names`) y comprobando al cargar el modelo que ambas coinciden antes de predecir. Un test que fije el orden del vector completa la protección.

## 5. Contrato de salida

Diseña la estructura que devolverá el módulo. Debe permitir responder:

- ¿Qué muestra se ha procesado?
- ¿Qué categoría se ha predicho?
- ¿Con qué confianza?
- ¿Qué versión de modelo y preprocesado se utilizaron?

In [ ]:
class WineQualityPredictionPydantic(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)

    sample_id: str = Field(min_length=1)  # qué muestra se ha procesado
    quality_band: Literal["acceptable", "excellent", "needs_review"]  # categoría predicha
    confidence: float = Field(ge=0.0, le=1.0)  # probabilidad de la clase predicha
    model_version: str = Field(min_length=1)  # versión del artefacto
    preprocessing_version: str = Field(min_length=1)  # versión de FEATURE_NAMES/preprocesado


# Ejemplo para red-001 (valores obtenidos con el módulo de la clase 2 y el modelo entregado).
example_output = WineQualityPredictionPydantic(
    sample_id="red-001",
    quality_band="needs_review",
    confidence=0.86,
    model_version="wine-quality-rf-v1",
    preprocessing_version="wine-red-features-v1",
)
print(example_output.model_dump())

# La salida también se valida: una categoría desconocida o una confianza > 1 se rechazan.
try:
    WineQualityPredictionPydantic(**{**example_output.model_dump(), "confidence": 1.5})
except ValidationError as error:
    print(error)

## 6. Demo del docente: observar la inferencia

El docente ejecutará el notebook resuelto y el comando de referencia. Mientras tanto, anota el recorrido:

```text
CSV → contrato → preprocesado → modelo cargado → predicción
```

No copies la implementación. Tu código de producción empieza en la clase 2 con el starter.

## Entrega

Entrega una captura o exportación con:

- La tabla de contrato.
- Las respuestas de las prácticas A, B y C.
- El orden de las 11 características.
- Un ejemplo de entrada inválida.
- Un contrato de salida.
- Tu plan de implementación para `contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py`.

## Lienzo de contrato y preprocesado (entrega)

### 1. Entrenamiento, artefacto y datos nuevos

| | Qué es | Dónde está | Quién lo usa |
|---|---|---|---|
| **Datos de entrenamiento** | Red Wine Quality (UCI) completo, con la columna `quality` | Sólo lo usa el profesorado; no se versiona | El entrenamiento, una vez |
| **Artefacto** | `wine_quality_classifier.joblib`: dict con `estimator` (RandomForest ya entrenado), `feature_names` y `model_version` | `starter/model/` | El módulo de inferencia, que sólo lo carga; no se reentrena |
| **Datos nuevos** | `inference_samples.csv`: 5 filas sin etiqueta, `sample_id` + 11 características | `assets/03-wine-quality/` | Entrada del CLI de inferencia |

### 2. Contrato de entrada

| # | Campo | Tipo | Unidad | Límite |
|---|---|---|---|---|
| — | `sample_id` | `str` | — | no vacío; no entra en el vector |
| 1 | `fixed_acidity` | `float` | g ácido tartárico / dm³ | 0–20 |
| 2 | `volatile_acidity` | `float` | g ácido acético / dm³ | 0–2 |
| 3 | `citric_acid` | `float` | g / dm³ | 0–2 |
| 4 | `residual_sugar` | `float` | g / dm³ | 0–20 |
| 5 | `chlorides` | `float` | g cloruro sódico / dm³ | 0–1 |
| 6 | `free_sulfur_dioxide` | `float` | mg / dm³ | 0–100 |
| 7 | `total_sulfur_dioxide` | `float` | mg / dm³ | 0–300 |
| 8 | `density` | `float` | g / cm³ | 0,98–1,01 |
| 9 | `ph` | `float` | adimensional | 2,5–4,5 |
| 10 | `sulphates` | `float` | g sulfato potásico / dm³ | 0–3 |
| 11 | `alcohol` | `float` | % vol. | 5–20 |

Cualquier columna fuera de esta tabla se rechaza (`extra="forbid"`).

### 3. Ejemplos

- **Válida** (`red-001`): `fixed_acidity=7.4, volatile_acidity=0.7, citric_acid=0.0, residual_sugar=1.9, chlorides=0.076, free_sulfur_dioxide=11.0, total_sulfur_dioxide=34.0, density=0.9978, ph=3.51, sulphates=0.56, alcohol=9.4`.
- **Inválida**: la misma fila con `unexpected_field="not allowed"` → `extra_forbidden`. Con `ph=99` → `less_than_equal`.

### 4. Orden del vector

`fixed_acidity, volatile_acidity, citric_acid, residual_sugar, chlorides, free_sulfur_dioxide, total_sulfur_dioxide, density, ph, sulphates, alcohol` — definido una vez en `FEATURE_NAMES` y comprobado contra `feature_names` del artefacto.

### 5. Criterios de aceptación para el script de la clase 2

1. `uv run python -m model_inference.predict_file --input assets/inference_samples.csv --output .tmp/wine_predictions.csv` termina con código 0 y escribe una fila por muestra con `sample_id, quality_band, confidence, model_version, preprocessing_version`.
2. Una fila con una columna extra, un campo que falta o un valor fuera de rango termina con código 2, un mensaje que nombra el campo y la muestra, y **no** deja ningún CSV de salida, aunque el error esté en una fila posterior.
3. Un artefacto incompleto, dañado o con `feature_names` en otro orden se rechaza al cargarlo, antes de predecir, con código 2 y sin traceback.

### Plan de implementación

- `contracts.py`: `WineQualityRequest` (tabla del punto 2, `extra="forbid"`) y `WineQualityPrediction` (contrato de salida de la sección 5).
- `preprocess.py`: `FEATURE_NAMES`, `PREPROCESSING_VERSION` y `preprocess_wine_request()`, que construye el vector en ese orden.
- `inference.py`: `load_wine_quality_model()` comprueba que el artefacto es un dict con `estimator` (con `predict` y `predict_proba`), `feature_names` igual a `FEATURE_NAMES` y `model_version`; `infer_wine_quality()` devuelve una `WineQualityPrediction` validada.
- `predict_file.py`: valida todas las filas antes de predecir, captura los errores de contrato y de artefacto (código 2) y sólo escribe el CSV si todo ha ido bien.